<h1 style="text-align:center">Face Spoofing Detection</h1>

<h5 style="text-align:center">Submitted for DAC FindIT 2026 Competition</h5>

## Table of Contents

1. [Introduction](#Introduction)
2. [Configuration](#Configuration)
3. [Data Exploration & Cleaning](#Data-Exploration-&-Cleaning)
4. [Preprocessing & Augmentation](#Preprocessing-&-Augmentation)
5. [Modeling Architecture](#Modeling-Architecture)
6. [Training Functions](#Training-Functions)
7. [Model Training & Validation](#Model-Training-&-Validation)
8. [Evaluation](#Evaluation)
9. [Inference & Submission](#Inference-&-Submission)
10. [Conclusion](#Conclusion)

# Introduction

Sistem face recognition sudah banyak dipakai di berbagai sektor, mulai dari pembukaan kunci perangkat, absensi, sampai verifikasi identitas di layanan keuangan. Tapi sistem ini masih bisa dikelabui oleh serangan **face spoofing**, yaitu upaya menipu pengenalan wajah pakai media visual seperti foto cetak, tampilan layar, masker, atau replika wajah.

Kompetisi ini menantang peserta untuk membangun model yang bisa mengklasifikasikan citra wajah ke dalam 6 kategori secara akurat dan mengidentifikasi berbagai bentuk serangan spoofing.

### Metrics: Macro F1-Score

Performa model diukur pakai **Macro F1-Score**, yaitu rata-rata harmonis dari precision dan recall yang dihitung per kelas lalu dirata-ratakan. Metrik ini cocok karena memberi bobot yang sama ke setiap kelas, tidak peduli jumlah sampelnya banyak atau sedikit.

$$F1_{macro} = \frac{1}{C} \sum_{i=1}^{C} \frac{2 \cdot P_i \cdot R_i}{P_i + R_i}$$

### Kelas Target

| Kelas | Deskripsi |
|:---|:---|
| **realperson** | Citra wajah asli tanpa manipulasi |
| **fake_printed** | Serangan menggunakan foto cetak (print attack) |
| **fake_screen** | Serangan melalui tampilan layar digital |
| **fake_mask** | Serangan menggunakan masker wajah / silikon |
| **fake_mannequin** | Serangan menggunakan replika wajah (mannequin) |
| **fake_unknown** | Jenis serangan lain yang tidak terdefinisi |

### Pendekatan

Pipeline yang dibangun:
1. **Data cleaning**: hapus duplikat lintas kelas dan koreksi label yang salah
2. **Multi-backbone ensemble**: gabungkan 3 arsitektur yang saling melengkapi (ConvNeXt, DINOv2, Swin)
3. **Test-Time Augmentation (TTA)**: tingkatkan stabilitas prediksi saat inferensi
4. **Weighted ensemble**: gabungkan probabilitas dari ketiga model

# Configuration

### Colab Setup

In [ ]:
from google.colab import userdata
import os

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

In [ ]:
!kaggle competitions download -c data-analytics-competition-dac-find-it-2026

100% 1.17G/1.17G [00:03<00:00, 336MB/s]



In [ ]:
!unzip data-analytics-competition-dac-find-it-2026.zip -d data

Archive:  data-analytics-competition-dac-find-it-2026.zip
  inflating: data/samplesubmission.csv  
  inflating: data/test/test_001.jpg  
  inflating: data/test/test_002.jpg  
  inflating: data/test/test_003.jpg  
  inflating: data/test/test_004.jpg  
  inflating: data/test/test_005.jpg  
  inflating: data/test/test_006.jpg  
  inflating: data/test/test_007.jpg  
  inflating: data/test/test_008.jpg  
  inflating: data/test/test_009.jpg  
  inflating: data/test/test_010.jpg  
  inflating: data/test/test_011.jpg  
  inflating: data/test/test_012.jpg  
  inflating: data/test/test_013.jpg  
  inflating: data/test/test_014.jpg  
  inflating: data/test/test_015.jpg  
  inflating: data/test/test_016.jpg  
  inflating: data/test/test_017.jpg  
  inflating: data/test/test_018.jpg  
  inflating: data/test/test_019.jpg  
  inflating: data/test/test_020.jpeg  
  inflating: data/test/test_021.jpg  
  inflating: data/test/test_022.jpg  
  inflating: data/test/test_023.jpg  
  inflating: data/test/tes

### Environment Setup

In [ ]:
!pip install -q timm albumentations 2>/dev/null

In [ ]:
# FocalLoss removed for V28
import os, gc, random, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(42)
assert torch.cuda.is_available()
device = torch.device('cuda')
USE_AMP = True
print(f'GPU: {torch.cuda.get_device_name(0)}')



GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


### Paths, Training & Model Configuration

Tiga backbone dipilih karena masing-masing menangkap aspek yang berbeda dari citra wajah:
- **ConvNeXt-Large**: arsitektur CNN modern yang bagus untuk menangkap tekstur lokal seperti pola cetak atau piksel layar
- **DINOv2-Large**: Vision Transformer dengan self-supervised pretraining di 142M gambar, kuat di representasi global
- **Swin-Large**: hierarchical ViT yang bisa menangkap hubungan spatial di berbagai skala lewat shifted window attention

Ketiga model ini di-ensemble supaya kelemahan satu model bisa ditutupi oleh model lain.

In [ ]:
DATA_DIR = Path('/content/data/')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
OUTPUT_DIR = Path('/content/')
NUM_CLASSES = 6
N_FOLDS = 5
IMG_SIZE = 224
SEEDS = [42]  # Single seed (V26 - proven best)

MODELS = {
    'convnext': {
        'name': 'convnext_large.fb_in22k_ft_in1k',
        'lr': 1e-4, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'relu', 'patience': 5, 'weight': 2.0,
    },
    'dinov2': {
        'name': 'vit_large_patch14_dinov2.lvd142m',
        'lr': 5e-5, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'gelu', 'patience': 5, 'weight': 1.0,
    },
    'swin': {
        'name': 'swin_large_patch4_window7_224.ms_in22k_ft_in1k',
        'lr': 1e-4, 'epochs': 20, 'batch': 16, 'freeze': 2,
        'head_type': 'relu', 'patience': 5, 'weight': 1.0,
    },
}

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(f'Models: {list(MODELS.keys())}')
print(f'Seeds: {SEEDS}')
print(f'Total groups: {len(MODELS) * len(SEEDS)}')

# Cross-class duplicate files to REMOVE (same image, different labels = poison)
DUPLICATE_REMOVE = {
    'mannequin_009.jpg', 'mannequin_024.jpg', 'mannequin_056.jpg',
    'mannequin_065.jpg', 'mannequin_069.jpg', 'mannequin_104.jpg',
    'mannequin_122.jpg', 'mannequin_146.jpg', 'mannequin_154.jpg',
    'mannequin_171.jpg', 'mannequin_192.jpg',
    'mask_009.jpg', 'mask_107.jpg', 'mask_122.jpg', 'mask_150.jpg',
    'mask_176.jpg', 'mask_257.jpg', 'mask_270.jpg',
    'screen_076.jpg', 'screen_216.jpg',
    'printed_007.jpeg', 'printed_012.jpeg', 'printed_021.jpg',
    'printed_027.jpg', 'printed_029.jpg', 'printed_041.jpg',
    'printed_068.jpeg', 'printed_071.jpeg', 'printed_073.jpeg',
    'printed_086.jpg',
    'unknown_009.jpg', 'unknown_051.jpg', 'unknown_110.jpeg',
    'unknown_235.jpg',
}


Models: ['convnext', 'dinov2', 'swin']
Seeds: [42]
Total groups: 3


# Data Exploration & Cleaning

### Data Overview

Dataset terdiri dari citra wajah yang dikelompokkan ke dalam 6 kategori. Setelah dieksplorasi, ditemukan dua masalah utama di data training:

1. **Cross-class duplicates**: ada 34 gambar yang sama persis tapi muncul di kelas berbeda. Ini bisa bikin model bingung karena dapat sinyal yang kontradiktif, jadi gambar-gambar ini dihapus.

2. **Label noise**: sekitar 189 gambar punya label yang salah berdasarkan review visual. Contohnya gambar wajah asli yang masuk folder fake_mask, atau sebaliknya. Label ini dikoreksi untuk meningkatkan kualitas data.

Gambar yang tidak jelas kelasnya ditandai REMOVE dan dikeluarkan dari training.

In [ ]:
CORRECTIONS = {
    'screen_009.jpg': 'fake_mannequin', 'screen_012.jpg': 'realperson',
    'screen_014.jpg': 'fake_mannequin', 'screen_018.jpg': 'realperson',
    'screen_020.jpg': 'realperson', 'screen_022.jpg': 'realperson',
    'screen_041.jpg': 'realperson', 'screen_043.jpg': 'fake_mannequin',
    'screen_044.jpg': 'realperson', 'screen_045.jpg': 'fake_mannequin',
    'screen_048.jpg': 'fake_mannequin', 'screen_050.jpg': 'realperson',
    'screen_057.jpg': 'realperson', 'screen_061.jpg': 'fake_mannequin',
    'screen_067.jpg': 'fake_mannequin', 'screen_073.jpg': 'realperson',
    'screen_076.jpg': 'fake_mannequin', 'screen_084.jpg': 'realperson',
    'screen_088.jpg': 'fake_mannequin', 'screen_105.jpg': 'fake_mannequin',
    'screen_120.jpg': 'fake_mannequin', 'screen_128.jpg': 'fake_mannequin',
    'screen_131.jpg': 'fake_mannequin', 'screen_132.jpg': 'fake_mannequin',
    'screen_148.jpg': 'realperson', 'screen_156.jpg': 'fake_mannequin',
    'screen_160.jpg': 'fake_mannequin', 'screen_168.jpg': 'realperson',
    'screen_173.jpg': 'fake_mannequin', 'screen_176.jpg': 'fake_mannequin',
    'screen_177.jpg': 'fake_mannequin', 'screen_190.jpg': 'fake_mannequin',
    'screen_205.JPG': 'fake_mannequin', 'screen_209.jpg': 'fake_mannequin',
    'screen_216.jpg': 'REMOVE', 'screen_217.jpg': 'realperson',
    'screen_220.jpg': 'realperson', 'screen_225.jpg': 'fake_mannequin',
    'screen_053.jpg': 'fake_mannequin',
    'printed_007.jpeg': 'REMOVE', 'printed_009.jpeg': 'fake_unknown',
    'printed_012.jpeg': 'REMOVE', 'printed_015.jpeg': 'fake_unknown',
    'printed_020.jpeg': 'fake_unknown', 'printed_034.jpeg': 'fake_unknown',
    'printed_035.jpeg': 'fake_unknown', 'printed_048.jpeg': 'fake_unknown',
    'printed_068.jpeg': 'REMOVE', 'printed_071.jpeg': 'REMOVE',
    'printed_072.jpeg': 'fake_unknown', 'printed_073.jpeg': 'REMOVE',
    'printed_081.jpeg': 'fake_unknown', 'printed_082.jpeg': 'fake_unknown',
    'printed_085.jpeg': 'fake_unknown', 'printed_095.jpeg': 'fake_unknown',
    'printed_099.jpeg': 'fake_unknown', 'printed_102.jpeg': 'fake_unknown',
    'printed_103.jpeg': 'fake_unknown', 'printed_106.jpeg': 'fake_unknown',
    'printed_002.jpeg': 'fake_unknown', 'printed_075.jpg': 'fake_mask',
    'mask_009.jpg': 'fake_printed', 'mask_018.jpeg': 'fake_unknown',
    'mask_037.jpg': 'fake_printed', 'mask_040.jpg': 'fake_printed',
    'mask_053.jpg': 'fake_printed', 'mask_055.jpg': 'fake_printed',
    'mask_061.jpeg': 'fake_unknown', 'mask_100.jpeg': 'fake_unknown',
    'mask_103.jpg': 'fake_printed', 'mask_107.jpg': 'fake_printed',
    'mask_123.jpg': 'fake_printed', 'mask_134.jpg': 'fake_printed',
    'mask_146.jpg': 'REMOVE', 'mask_148.jpg': 'realperson',
    'mask_150.jpg': 'fake_printed', 'mask_176.jpg': 'fake_printed',
    'mask_192.jpg': 'fake_printed', 'mask_193.jpg': 'fake_printed',
    'mask_197.jpg': 'fake_printed', 'mask_198.jpg': 'fake_printed',
    'mask_203.jpg': 'realperson', 'mask_204.jpeg': 'fake_unknown',
    'mask_223.jpg': 'fake_printed', 'mask_242.jpg': 'realperson',
    'mask_246.jpeg': 'fake_unknown', 'mask_247.jpg': 'fake_printed',
    'mask_249.jpg': 'fake_printed', 'mask_257.jpg': 'fake_printed',
    'mask_259.jpg': 'fake_printed',
    'mannequin_009.jpg': 'REMOVE', 'mannequin_024.jpg': 'REMOVE',
    'mannequin_047.jpg': 'fake_printed', 'mannequin_056.jpg': 'REMOVE',
    'mannequin_061.jpg': 'fake_printed', 'mannequin_065.jpg': 'REMOVE',
    'mannequin_069.jpg': 'fake_printed', 'mannequin_096.jpg': 'realperson',
    'mannequin_104.jpg': 'fake_printed', 'mannequin_122.jpg': 'fake_printed',
    'mannequin_127.jpg': 'fake_printed', 'mannequin_146.jpg': 'fake_printed',
    'mannequin_154.jpg': 'REMOVE', 'mannequin_171.jpg': 'REMOVE',
    'mannequin_190.jpg': 'realperson', 'mannequin_192.jpg': 'fake_printed',
    'mannequin_215.jpg': 'fake_printed', 'mannequin_216.jpg': 'realperson',
    'unknown_002.jpg': 'realperson', 'unknown_004.jpg': 'realperson',
    'unknown_005.jpg': 'fake_mask', 'unknown_009.jpg': 'REMOVE',
    'unknown_014.jpg': 'realperson', 'unknown_051.jpg': 'fake_mask',
    'unknown_064.jpg': 'fake_mask', 'unknown_070.jpg': 'realperson',
    'unknown_076.jpeg': 'fake_mannequin', 'unknown_081.jpg': 'realperson',
    'unknown_082.jpg': 'realperson', 'unknown_099.jpg': 'realperson',
    'unknown_110.jpeg': 'REMOVE', 'unknown_136.jpg': 'realperson',
    'unknown_163.jpg': 'fake_mask', 'unknown_225.jpg': 'fake_printed',
    'unknown_235.jpg': 'fake_mask', 'unknown_236.jpg': 'fake_mask',
    'unknown_239.jpg': 'realperson', 'unknown_240.jpg': 'realperson',
    'unknown_264.jpg': 'fake_printed', 'unknown_278.jpg': 'fake_mask',
    'unknown_280.jpg': 'realperson', 'unknown_290.jpg': 'fake_printed',
    'unknown_291.jpg': 'realperson', 'unknown_292.jpg': 'fake_mask',
    'unknown_314.jpg': 'fake_mask', 'unknown_332.jpg': 'fake_printed',
    'unknown_336.jpg': 'fake_mask',
    'unknown_222.jpg': 'fake_mask', 'unknown_214.jpg': 'realperson',
    'unknown_224.jpg': 'realperson', 'unknown_326.jpeg': 'realperson',
    'real_043.jpeg': 'fake_unknown', 'real_067.jpeg': 'fake_unknown',
    'real_071.jpg': 'fake_printed', 'real_075.jpeg': 'fake_unknown',
    'real_087.jpeg': 'fake_unknown', 'real_094.jpg': 'fake_mask',
    'real_110.jpg': 'fake_screen', 'real_132.jpg': 'fake_printed',
    'real_139.jpg': 'fake_screen', 'real_166.jpg': 'fake_mask',
    'real_168.jpeg': 'fake_unknown', 'real_170.jpeg': 'fake_unknown',
    'real_181.jpg': 'fake_mask', 'real_185.jpeg': 'fake_unknown',
    'real_187.jpeg': 'fake_unknown', 'real_192.jpeg': 'fake_unknown',
    'real_197.jpg': 'fake_printed', 'real_201.jpg': 'fake_printed',
    'real_233.jpg': 'fake_printed', 'real_237.jpeg': 'fake_printed',
    'real_260.jpg': 'fake_mask', 'real_266.jpeg': 'fake_unknown',
    'real_272.jpg': 'fake_screen', 'real_283.jpeg': 'fake_unknown',
    'real_294.jpeg': 'fake_unknown', 'real_298.jpg': 'fake_screen',
    'real_301.jpeg': 'fake_unknown', 'real_317.jpg': 'fake_printed',
    'real_330.jpg': 'fake_screen', 'real_346.jpg': 'fake_mask',
    'real_349.jpeg': 'fake_unknown', 'real_354.jpeg': 'fake_unknown',
    'real_357.jpg': 'fake_screen', 'real_365.jpeg': 'fake_unknown',
    'real_382.jpg': 'fake_printed', 'real_388.jpg': 'fake_screen',
    'real_395.jpeg': 'fake_unknown', 'real_402.jpeg': 'fake_unknown',
    'real_409.jpeg': 'fake_unknown', 'real_422.jpg': 'fake_printed',
    'real_433.jpeg': 'fake_unknown', 'real_435.jpg': 'fake_mask',
    'real_437.jpeg': 'fake_unknown', 'real_438.jpeg': 'fake_unknown',
    'real_447.jpg': 'fake_screen', 'real_448.jpeg': 'fake_unknown',
    'real_457.jpg': 'fake_screen', 'real_427.jpeg': 'fake_unknown',
}

def build_train_df():
    rows = []
    for cls in CLASS_NAMES:
        for p in sorted((TRAIN_DIR / cls).glob('*')):
            if p.suffix.lower() not in ['.jpg','.jpeg','.png','.bmp','.webp']: continue
            fname = p.name
            if fname in DUPLICATE_REMOVE:
                continue
            if fname in CORRECTIONS:
                nl = CORRECTIONS[fname]
                if nl == 'REMOVE': continue
                rows.append({'filepath': str(p), 'label': nl, 'label_idx': CLASS_TO_IDX[nl]})
            else:
                rows.append({'filepath': str(p), 'label': cls, 'label_idx': CLASS_TO_IDX[cls]})
    return pd.DataFrame(rows)

def build_test_df():
    rows = []
    for p in sorted(TEST_DIR.glob('*')):
        if p.suffix.lower() in ['.jpg','.jpeg','.png','.bmp','.webp']:
            rows.append({'filepath': str(p), 'id': p.stem})
    return pd.DataFrame(rows)

train_df = build_train_df()
test_df = build_test_df()
print(f'Train: {len(train_df)} | Test: {len(test_df)}')



Train: 1617 | Test: 404


# Preprocessing & Augmentation

Augmentasi dirancang untuk menyimulasikan variasi kondisi yang mungkin ditemui saat inferensi:
- Geometri: random crop, horizontal flip, rotasi
- Kualitas: Gaussian blur, motion blur, noise (simulasi kamera berkualitas rendah)
- Pencahayaan: color jitter

Saat inferensi, Test-Time Augmentation (TTA) dengan 4 varian dipakai untuk meningkatkan stabilitas prediksi. Varian yang digunakan: original, flip horizontal, zoom in, dan resize langsung.

Setiap backbone punya classification head yang sama: LayerNorm, Dropout(0.3), Linear, Activation, Dropout(0.15), Linear(6).

In [ ]:

def get_train_transforms(img_size):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size,img_size), scale=(0.7,1.0)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
        A.OneOf([A.GaussianBlur(blur_limit=(3,5)), A.MotionBlur(blur_limit=(3,5))], p=0.2),
        A.OneOf([A.GaussNoise(), A.ISONoise()], p=0.2),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size+32, img_size+32),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ToTensorV2(),
    ])

def get_tta_transforms(img_size):
    N = A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    return [
        get_val_transforms(img_size),
        A.Compose([A.Resize(img_size+32,img_size+32), A.CenterCrop(img_size,img_size),
                   A.HorizontalFlip(p=1.0), N, ToTensorV2()]),
        A.Compose([A.Resize(img_size+64,img_size+64), A.CenterCrop(img_size,img_size), N, ToTensorV2()]),
        A.Compose([A.Resize(img_size,img_size), N, ToTensorV2()]),
    ]

class FaceDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try: img = np.array(Image.open(row['filepath']).convert('RGB'))
        except: img = np.zeros((224,224,3), dtype=np.uint8)
        if self.transform: img = self.transform(image=img)['image']
        return img if self.is_test else (img, row['label_idx'])

class SpoofModel(nn.Module):
    def __init__(self, model_name, head_type='relu', pretrained=True):
        super().__init__()
        kwargs = dict(pretrained=pretrained, num_classes=0)
        if 'vit' in model_name or 'dino' in model_name:
            kwargs['img_size'] = IMG_SIZE
        self.backbone = timm.create_model(model_name, **kwargs)
        nf = self.backbone.num_features
        act = nn.GELU() if head_type == 'gelu' else nn.ReLU(inplace=True)
        self.head = nn.Sequential(
            nn.LayerNorm(nf) if head_type == 'gelu' else nn.Identity(),
            nn.Dropout(0.3), nn.Linear(nf, 256), act,
            nn.Dropout(0.15), nn.Linear(256, NUM_CLASSES)
        )
    def forward(self, x): return self.head(self.backbone(x))
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

# Modeling Architecture

### Training Functions

Beberapa teknik regularisasi dipakai untuk menangani karakteristik dataset:

| Teknik | Alasan |
|:---|:---|
| **Mixup** (alpha=0.4) | Regularisasi dengan interpolasi linear antara gambar dan label |
| **Label Smoothing** (0.15) | Mengurangi overconfidence, apalagi di data yang labelnya belum tentu benar |
| **Class Weights** | Menangani ketidakseimbangan jumlah sampel antar kelas |
| **Early Stopping** (patience=5) | Menghentikan training kalau sudah tidak ada improvement |

In [ ]:
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def train_one_epoch(model, loader, criterion, optimizer, scaler, use_mixup=True):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            if use_mixup and np.random.random() > 0.5:
                mx, ya, yb, lam = mixup_data(images, targets)
                out = model(mx)
                loss = lam*criterion(out,ya) + (1-lam)*criterion(out,yb)
                labels.extend(ya.cpu().numpy())
            else:
                out = model(images)
                loss = criterion(out, targets)
                labels.extend(targets.cpu().numpy())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        preds.extend(out.detach().argmax(1).cpu().numpy())
    return total_loss/len(loader), f1_score(labels[:len(preds)], preds, average='macro')

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, preds, labels = 0.0, [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(images)
            loss = criterion(out, targets)
        total_loss += loss.item()
        preds.extend(torch.softmax(out.float(),1).argmax(1).cpu().numpy())
        labels.extend(targets.cpu().numpy())
    return total_loss/len(loader), f1_score(labels, preds, average='macro')

@torch.no_grad()
def predict_tta(model, test_df, tta_transforms, batch_size):
    model.eval()
    all_probs = []
    for transform in tta_transforms:
        ds = FaceDataset(test_df, transform=transform, is_test=True)
        loader = DataLoader(ds, batch_size=batch_size*2, shuffle=False, num_workers=2)
        probs = []
        for images in loader:
            images = images.to(device)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(images)
            probs.append(torch.softmax(out.float(),1).cpu().numpy())
        all_probs.append(np.concatenate(probs))
    return np.mean(all_probs, axis=0)

# Model Training & Validation

### Training Loop

Training dilakukan per model dengan stratified 5-fold cross validation. Fine-tuning pakai dua fase:
1. **Freeze phase** (2 epoch): hanya classification head yang dilatih, backbone tetap frozen
2. **Fine-tune phase**: backbone di-unfreeze dengan learning rate 10x lebih kecil dari head supaya representasi pretrained tidak rusak

Learning rate di-schedule pakai CosineAnnealingWarmRestarts.

In [ ]:
class_counts = train_df['label'].value_counts().sort_index()
class_weights = torch.tensor(
    [len(train_df)/(NUM_CLASSES*class_counts[c]) for c in CLASS_NAMES], dtype=torch.float32
).to(device)

all_probs = {}
all_oof_preds = []
all_oof_labels = []
best_global_f1 = 0.0
best_global_ckpt = None
t_global = time.time()

for seed in SEEDS:
    for arch_name, cfg in MODELS.items():
        group_key = f's{seed}_{arch_name}'
        print(f'\n{"#"*60}')
        print(f'# {group_key} ({cfg["name"]})')
        print(f'{"#"*60}')
        seed_everything(seed)

        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['label_idx'])):
            print(f'  Fold {fold+1}/{N_FOLDS}', end=' ', flush=True)
            t0 = time.time()

            train_ds = FaceDataset(train_df.iloc[train_idx], transform=get_train_transforms(IMG_SIZE))
            val_ds = FaceDataset(train_df.iloc[val_idx], transform=get_val_transforms(IMG_SIZE))
            train_loader = DataLoader(train_ds, batch_size=cfg['batch'], shuffle=True,
                                      num_workers=2, pin_memory=True, drop_last=True)
            val_loader = DataLoader(val_ds, batch_size=cfg['batch']*2, shuffle=False,
                                    num_workers=2, pin_memory=True)

            model = SpoofModel(cfg['name'], cfg['head_type']).to(device)
            model.freeze_backbone()
            criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
            optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)
            scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

            best_f1, patience_counter = 0.0, 0
            for epoch in range(cfg['epochs']):
                if epoch == cfg['freeze']:
                    model.unfreeze_backbone()
                    optimizer = optim.AdamW([
                        {'params': model.backbone.parameters(), 'lr': cfg['lr']*0.1},
                        {'params': model.head.parameters(), 'lr': cfg['lr']},
                    ], weight_decay=1e-4)
                    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-7)
                    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

                t_loss, t_f1 = train_one_epoch(model, train_loader, criterion, optimizer, scaler, epoch>=cfg['freeze'])
                v_loss, val_f1 = validate(model, val_loader, criterion)
                scheduler.step()
                print(f'    e{epoch+1} tF1={t_f1:.3f} vF1={val_f1:.3f}', flush=True)

                if val_f1 > best_f1:
                    best_f1 = val_f1; patience_counter = 0
                    torch.save(model.state_dict(), OUTPUT_DIR / f'{group_key}_f{fold}.pth')
                else:
                    patience_counter += 1
                    if patience_counter >= cfg['patience']: break

            fold_scores.append(best_f1)
            # Track best model globally
            if best_f1 > best_global_f1:
                best_global_f1 = best_f1
                best_global_ckpt = f'{group_key}_f{fold}.pth'
            # Collect OOF predictions
            model.load_state_dict(torch.load(OUTPUT_DIR / f'{group_key}_f{fold}.pth', weights_only=True))
            model.eval()
            with torch.no_grad():
                for images, targets in val_loader:
                    images = images.to(device)
                    with torch.cuda.amp.autocast(enabled=USE_AMP):
                        out = model(images)
                    all_oof_preds.extend(torch.softmax(out.float(),1).argmax(1).cpu().numpy())
                    all_oof_labels.extend(targets.numpy())
            print(f'    Best={best_f1:.4f} ({(time.time()-t0)/60:.1f}m)', flush=True)
            del model, optimizer, scheduler, scaler
            gc.collect(); torch.cuda.empty_cache()

        cv = np.mean(fold_scores)
        print(f'  CV: {cv:.4f}')

        # Predict test (predict per fold, delete checkpoint immediately to save disk)
        tta = get_tta_transforms(IMG_SIZE)
        group_probs = np.zeros((len(test_df), NUM_CLASSES))
        for fold in range(N_FOLDS):
            ckpt = OUTPUT_DIR / f'{group_key}_f{fold}.pth'
            model = SpoofModel(cfg['name'], cfg['head_type'], pretrained=False).to(device)
            model.load_state_dict(torch.load(ckpt, weights_only=True))
            group_probs += predict_tta(model, test_df, tta, cfg['batch'])
            del model; gc.collect(); torch.cuda.empty_cache()
            if str(ckpt.name) != best_global_ckpt:
                os.remove(ckpt)
            print(f'    Predicted fold {fold}', flush=True)
        group_probs /= N_FOLDS
        all_probs[group_key] = group_probs

print(f'\nTotal time: {(time.time()-t_global)/60:.0f} min')



############################################################
# s42_convnext (convnext_large.fb_in22k_ft_in1k)
############################################################
  Fold 1/5 

model.safetensors:   0%|          | 0.00/791M [00:00<?, ?B/s]

    e1 tF1=0.652 vF1=0.873
    e2 tF1=0.808 vF1=0.902
    e3 tF1=0.716 vF1=0.918
    e4 tF1=0.767 vF1=0.932
    e5 tF1=0.766 vF1=0.938
    e6 tF1=0.803 vF1=0.938
    e7 tF1=0.782 vF1=0.938
    e8 tF1=0.749 vF1=0.941
    e9 tF1=0.735 vF1=0.950
    e10 tF1=0.791 vF1=0.948
    e11 tF1=0.801 vF1=0.953
    e12 tF1=0.812 vF1=0.943
    e13 tF1=0.823 vF1=0.953
    e14 tF1=0.785 vF1=0.953
    e15 tF1=0.805 vF1=0.958
    e16 tF1=0.817 vF1=0.953
    e17 tF1=0.758 vF1=0.953
    e18 tF1=0.821 vF1=0.955
    e19 tF1=0.804 vF1=0.958
    e20 tF1=0.722 vF1=0.955
    Best=0.9579 (4.1m)
  Fold 2/5     e1 tF1=0.614 vF1=0.870
    e2 tF1=0.809 vF1=0.875
    e3 tF1=0.700 vF1=0.929
    e4 tF1=0.757 vF1=0.942
    e5 tF1=0.762 vF1=0.949
    e6 tF1=0.718 vF1=0.960
    e7 tF1=0.819 vF1=0.960
    e8 tF1=0.785 vF1=0.966
    e9 tF1=0.772 vF1=0.962
    e10 tF1=0.800 vF1=0.971
    e11 tF1=0.769 vF1=0.965
    e12 tF1=0.807 vF1=0.968
    e13 tF1=0.759 vF1=0.968
    e14 tF1=0.784 vF1=0.971
    e15 tF1=0.791 vF1=0.971
    

    e1 tF1=0.641 vF1=0.892
    e2 tF1=0.794 vF1=0.923
    e3 tF1=0.609 vF1=0.951
    e4 tF1=0.675 vF1=0.966
    e5 tF1=0.719 vF1=0.966
    e6 tF1=0.727 vF1=0.973
    e7 tF1=0.777 vF1=0.971
    e8 tF1=0.788 vF1=0.967
    e9 tF1=0.800 vF1=0.973
    e10 tF1=0.773 vF1=0.971
    e11 tF1=0.812 vF1=0.976
    e12 tF1=0.756 vF1=0.982
    e13 tF1=0.807 vF1=0.980
    e14 tF1=0.818 vF1=0.978
    e15 tF1=0.805 vF1=0.981
    e16 tF1=0.828 vF1=0.981
    e17 tF1=0.837 vF1=0.981
    Best=0.9818 (2.7m)
  CV: 0.9644
    Predicted fold 0
    Predicted fold 1
    Predicted fold 2
    Predicted fold 3
    Predicted fold 4

############################################################
# s42_dinov2 (vit_large_patch14_dinov2.lvd142m)
############################################################
  Fold 1/5 

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

    e1 tF1=0.556 vF1=0.844
    e2 tF1=0.816 vF1=0.892
    e3 tF1=0.736 vF1=0.950
    e4 tF1=0.788 vF1=0.944
    e5 tF1=0.741 vF1=0.955
    e6 tF1=0.803 vF1=0.965
    e7 tF1=0.806 vF1=0.960
    e8 tF1=0.775 vF1=0.937
    e9 tF1=0.739 vF1=0.956
    e10 tF1=0.790 vF1=0.970
    e11 tF1=0.792 vF1=0.970
    e12 tF1=0.814 vF1=0.963
    e13 tF1=0.823 vF1=0.970
    e14 tF1=0.782 vF1=0.971
    e15 tF1=0.815 vF1=0.965
    e16 tF1=0.817 vF1=0.971
    e17 tF1=0.773 vF1=0.971
    e18 tF1=0.840 vF1=0.968
    e19 tF1=0.792 vF1=0.961
    e20 tF1=0.741 vF1=0.960
    Best=0.9708 (3.3m)
  Fold 2/5     e1 tF1=0.565 vF1=0.839
    e2 tF1=0.806 vF1=0.892
    e3 tF1=0.718 vF1=0.951
    e4 tF1=0.742 vF1=0.954
    e5 tF1=0.773 vF1=0.972
    e6 tF1=0.746 vF1=0.976
    e7 tF1=0.833 vF1=0.981
    e8 tF1=0.793 vF1=0.962
    e9 tF1=0.766 vF1=0.973
    e10 tF1=0.816 vF1=0.978
    e11 tF1=0.772 vF1=0.979
    e12 tF1=0.800 vF1=0.973
    Best=0.9805 (2.0m)
  Fold 3/5     e1 tF1=0.531 vF1=0.881
    e2 tF1=0.814 vF1=0.889


model.safetensors:   0%|          | 0.00/788M [00:00<?, ?B/s]

    e1 tF1=0.643 vF1=0.871
    e2 tF1=0.811 vF1=0.907
    e3 tF1=0.718 vF1=0.916
    e4 tF1=0.766 vF1=0.940
    e5 tF1=0.746 vF1=0.947
    e6 tF1=0.811 vF1=0.940
    e7 tF1=0.779 vF1=0.944
    e8 tF1=0.739 vF1=0.950
    e9 tF1=0.729 vF1=0.946
    e10 tF1=0.800 vF1=0.948
    e11 tF1=0.793 vF1=0.955
    e12 tF1=0.808 vF1=0.955
    e13 tF1=0.819 vF1=0.955
    e14 tF1=0.775 vF1=0.955
    e15 tF1=0.804 vF1=0.953
    e16 tF1=0.814 vF1=0.955
    e17 tF1=0.754 vF1=0.955
    e18 tF1=0.840 vF1=0.951
    e19 tF1=0.793 vF1=0.955
    Best=0.9554 (3.2m)
  Fold 2/5     e1 tF1=0.629 vF1=0.863
    e2 tF1=0.816 vF1=0.875
    e3 tF1=0.635 vF1=0.925
    e4 tF1=0.739 vF1=0.946
    e5 tF1=0.768 vF1=0.955
    e6 tF1=0.755 vF1=0.955
    e7 tF1=0.733 vF1=0.957
    e8 tF1=0.804 vF1=0.966
    e9 tF1=0.790 vF1=0.965
    e10 tF1=0.776 vF1=0.977
    e11 tF1=0.808 vF1=0.977
    e12 tF1=0.761 vF1=0.974
    e13 tF1=0.800 vF1=0.977
    e14 tF1=0.743 vF1=0.980
    e15 tF1=0.789 vF1=0.977
    e16 tF1=0.777 vF1=0.974
    

# Evaluation

Performa dievaluasi pakai prediksi out-of-fold (OOF) dari cross validation. Setiap sampel training diprediksi oleh model yang tidak melihatnya saat training, jadi estimasi performanya tidak bias.

Model terbaik berdasarkan validation F1-score disimpan sebagai best_model.pth.

In [ ]:
# OOF Evaluation
oof_f1 = f1_score(all_oof_labels, all_oof_preds, average='macro')
print(f'OOF Macro F1-Score: {oof_f1:.4f}')
print()
print('Classification Report:')
print(classification_report(all_oof_labels, all_oof_preds, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(all_oof_labels, all_oof_preds)
print('Confusion Matrix:')
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(cm_df)

# Per-class accuracy
print()
for i, cls in enumerate(CLASS_NAMES):
    total = cm[i].sum()
    correct = cm[i][i]
    print(f'  {cls:20s}: {correct}/{total} ({correct/total:.1%})')

# Save best model
import shutil
best_src = OUTPUT_DIR / best_global_ckpt
best_dst = OUTPUT_DIR / 'best_model.pth'
if best_src.exists():
    shutil.copy2(best_src, best_dst)
    os.remove(best_src)
    print(f'\nModel terbaik disimpan: best_model.pth (fold F1={best_global_f1:.4f})')


OOF Macro F1-Score: 0.9689

Classification Report:
                precision    recall  f1-score   support

fake_mannequin       0.97      0.98      0.97       690
     fake_mask       0.93      0.96      0.94       789
  fake_printed       0.95      0.99      0.97       369
   fake_screen       0.98      0.97      0.97       597
  fake_unknown       0.99      0.97      0.98      1050
    realperson       0.98      0.97      0.97      1356

      accuracy                           0.97      4851
     macro avg       0.97      0.97      0.97      4851
  weighted avg       0.97      0.97      0.97      4851

Confusion Matrix:
                fake_mannequin  fake_mask  fake_printed  fake_screen  \
fake_mannequin             673          6             0            0   
fake_mask                    9        758             9            0   
fake_printed                 0          3           366            0   
fake_screen                  0          2             4          578   
fake_unk

# Inference & Submission

Probabilitas dari ketiga model digabung pakai weighted average. Weight-nya ditentukan berdasarkan performa masing-masing model di cross validation.

Submission per model juga disimpan untuk analisis lebih lanjut.

In [ ]:
final_probs = np.zeros((len(test_df), NUM_CLASSES))
total_weight = 0

for key, probs in all_probs.items():
    arch = key.split('_', 1)[1]
    w = MODELS[arch]['weight']
    final_probs += probs * w
    total_weight += w
    preds = [IDX_TO_CLASS[p] for p in probs.argmax(axis=1)]
    print(f'{key} (w={w}): {pd.Series(preds).value_counts().sort_index().to_dict()}')
    # Save per-group probs
    np.save(OUTPUT_DIR / f'probs_{key}.npy', probs)
    # Save per-model CSV for offline analysis
    model_preds = [IDX_TO_CLASS[p] for p in probs.argmax(axis=1)]
    pd.DataFrame({
        'id': test_df['id'], 'label': model_preds,
        **{c: probs[:,i] for i,c in enumerate(CLASS_NAMES)}
    }).to_csv(OUTPUT_DIR / f'submission_{arch}.csv', index=False)
    print(f'  -> Saved submission_{arch}.csv')


final_probs /= total_weight
final_preds = [IDX_TO_CLASS[p] for p in final_probs.argmax(axis=1)]

submission = pd.DataFrame({'id': test_df['id'], 'label': final_preds})
submission = submission.sort_values('id').reset_index(drop=True)
submission.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'\nEnsemble: {pd.Series(final_preds).value_counts().sort_index().to_dict()}')
print(f'Saved submission.csv ({len(submission)} rows)')

# Save probabilities for offline threshold tuning
np.save(OUTPUT_DIR / 'final_probs.npy', final_probs)
# Save test IDs + class names for reference
pd.DataFrame({'id': test_df['id'].values, **{c: final_probs[:,i] for i,c in enumerate(CLASS_NAMES)}}).to_csv(OUTPUT_DIR / 'test_probabilities.csv', index=False)
print('Saved final_probs.npy + test_probabilities.csv')
print(f'\nClass names order: {CLASS_NAMES}')

s42_convnext (w=2.0): {'fake_mannequin': 52, 'fake_mask': 66, 'fake_printed': 60, 'fake_screen': 71, 'fake_unknown': 49, 'realperson': 106}
  -> Saved submission_convnext.csv
s42_dinov2 (w=1.0): {'fake_mannequin': 49, 'fake_mask': 68, 'fake_printed': 62, 'fake_screen': 72, 'fake_unknown': 49, 'realperson': 104}
  -> Saved submission_dinov2.csv
s42_swin (w=1.0): {'fake_mannequin': 51, 'fake_mask': 65, 'fake_printed': 62, 'fake_screen': 73, 'fake_unknown': 49, 'realperson': 104}
  -> Saved submission_swin.csv

Ensemble: {'fake_mannequin': 50, 'fake_mask': 68, 'fake_printed': 59, 'fake_screen': 72, 'fake_unknown': 49, 'realperson': 106}
Saved submission.csv (404 rows)
Saved final_probs.npy + test_probabilities.csv

Class names order: ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']


### Confidence Analysis

Prediksi dengan confidence rendah menunjukkan gambar yang ambigu atau ada di batas keputusan antar kelas. Analisis ini berguna untuk tahu di mana model masih kesulitan.

In [ ]:
max_probs = final_probs.max(axis=1)
print(f'Confidence statistics:')
print(f'  Mean:   {max_probs.mean():.4f}')
print(f'  Median: {np.median(max_probs):.4f}')
print(f'  Min:    {max_probs.min():.4f}')
print(f'  < 0.5:  {(max_probs < 0.5).sum()} images')
print(f'  < 0.7:  {(max_probs < 0.7).sum()} images')
print(f'  > 0.9:  {(max_probs > 0.9).sum()} images')

low_conf = np.where(max_probs < 0.5)[0]
if len(low_conf) > 0:
    print(f'\nLow confidence predictions (< 0.5):')
    for idx in low_conf[:15]:
        tid = test_df.iloc[idx]['id']
        pred = final_preds[idx]
        conf = max_probs[idx]
        top2_idx = final_probs[idx].argsort()[-2:][::-1]
        top2 = [(IDX_TO_CLASS[i], final_probs[idx][i]) for i in top2_idx]
        print(f'  {tid}: {pred} ({conf:.3f}) | 2nd: {top2[1][0]} ({top2[1][1]:.3f})')


Confidence statistics:
  Mean:   0.7761
  Median: 0.8027
  Min:    0.3224
  < 0.5:  27 images
  < 0.7:  85 images
  > 0.9:  63 images

Low confidence predictions (< 0.5):
  test_003: realperson (0.420) | 2nd: fake_mask (0.341)
  test_024: fake_unknown (0.381) | 2nd: fake_printed (0.259)
  test_049: fake_mask (0.443) | 2nd: fake_printed (0.422)
  test_065: fake_mask (0.465) | 2nd: fake_mannequin (0.282)
  test_094: fake_mask (0.429) | 2nd: fake_mannequin (0.230)
  test_122: fake_screen (0.499) | 2nd: realperson (0.171)
  test_149: realperson (0.468) | 2nd: fake_mask (0.174)
  test_151: realperson (0.411) | 2nd: fake_unknown (0.165)
  test_156: fake_unknown (0.485) | 2nd: fake_printed (0.134)
  test_160: realperson (0.450) | 2nd: fake_mask (0.224)
  test_167: fake_mask (0.435) | 2nd: fake_printed (0.401)
  test_168: fake_mask (0.420) | 2nd: fake_printed (0.299)
  test_187: realperson (0.373) | 2nd: fake_printed (0.319)
  test_207: realperson (0.459) | 2nd: fake_printed (0.286)
  test_210

# Conclusion

### Summary

Notebook ini menyajikan pipeline lengkap untuk face spoofing detection pakai multi-backbone ensemble (ConvNeXt-Large, DINOv2-Large, Swin-Large).

### Key Findings

| Finding | Impact |
|:---|:---|
| Dataset punya sekitar 189 label noise dan 34 duplikat lintas kelas | Data cleaning meningkatkan kualitas training secara signifikan |
| Ensemble 3 backbone lebih stabil dari model tunggal | Tiap arsitektur menangkap aspek berbeda dari spoofing |
| TTA dengan 4 varian meningkatkan konsistensi prediksi | Mengurangi variance di gambar yang ambigu |
| Class weighting penting untuk kelas minoritas | Tanpa weighting, kelas kecil sering salah klasifikasi |